# Safety distillation (Qwen2.5-3B-Instruct → Qwen2.5-0.5B-Instruct)


## Data loading

In [ ]:
import os
import re
import gc
import random
import tarfile
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers
import peft
import accelerate
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Check versions and GPU
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
    print("BF16 natively supported:", torch.cuda.is_bf16_supported(including_emulation=False))
else:
    print("no gpu available")

In [ ]:
# Config: model names and number format
STUDENT_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TEACHER_ID = "Qwen/Qwen2.5-3B-Instruct"
GUARD_ID   = "Qwen/Qwen3Guard-Gen-0.6B"

# bf16 only on GPUs with native support (Ampere+, e.g. A100/L4); fp16 on a T4 (emulated bf16 is very slow)
NATIVE_BF16 = torch.cuda.get_device_capability()[0] >= 8
dtype = torch.bfloat16 if NATIVE_BF16 else torch.float16
print("using dtype:", dtype)

# Folder where results are saved
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/safety_kd"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Helpers

# for results to be reproducible
def seed_all(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(0)

# to release GPU memory after `del some_model`
def free():
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Download and extract the data
tgz_url = "http://www.linguist.univ-paris-diderot.fr/~mcandito/Ens/safety-kd-data.tgz"
tgz_filename = "safety-kd-data.tgz"
extract_dir = "."

if not os.path.exists(tgz_filename):
    print(f"Downloading {tgz_filename}...")
    !wget -O {tgz_filename} "{tgz_url}"
else:
    print(f"{tgz_filename} already exists.")

try:
    with tarfile.open(tgz_filename, "r:gz") as tar_ref:
        tar_ref.extractall(extract_dir, filter="data")
    print("Extraction complete.")
except Exception as e:
    print(f"Error extracting tar file: {e}")

In [ ]:
# Load the 4 tables
base = "./safety-kd-data"
train  = pd.read_json(f"{base}/BTfraction_train.jsonl", lines=True)   # 2000 prompts + BT / teacher / student answers
val    = pd.read_json(f"{base}/BTfraction_val.jsonl", lines=True)     # 200 prompts (validation loss only)
bteval = pd.read_json(f"{base}/evaluation/BTeval.json")               # 700 harmful eval prompts + teacher / student answers
xstest = pd.read_json(f"{base}/evaluation/XSTest.json")               # 450 eval prompts (over-refusal) + teacher / student answers

# Same "prompt type" column everywhere, used by the metrics
train["ptype"]  = np.where(train["benign"], "benign", "harmful")
bteval["ptype"] = "harmful"
xstest["ptype"] = np.where(xstest["label"] == "safe", "benign", "harmful")

print(train.ptype.value_counts(), xstest.ptype.value_counts(), sep="\n")
# share of answers cut at 256 tokens
print("truncated (train) teacher/student:", train.teacher_truncated.mean(), train.student_truncated.mean())

In [ ]:
# Look at the data
print("BTfraction_train:"); display(train.head())
print("\nEvaluation : BTeval:"); display(bteval.head())
print("\nEvaluation : XSTest:"); display(xstest.head())

# Guard + metrics

In [ ]:
# Load the guard
guard_tok = AutoTokenizer.from_pretrained(GUARD_ID, padding_side="left")
guard = AutoModelForCausalLM.from_pretrained(GUARD_ID, dtype=torch.float32).to("cuda").eval()

In [ ]:
# Judge ONE (prompt, answer) pair -> raw text from the guard
def judge_one(prompt, answer):
    messages = [{"role": "user", "content": prompt},
                {"role": "assistant", "content": answer}]
    text = guard_tok.apply_chat_template(messages, tokenize=False)
    inputs = guard_tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = guard.generate(**inputs, max_new_tokens=32, do_sample=False)
    return guard_tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
# Test on one example
row = train.iloc[0]
print("PROMPT:", row.prompt)
print("ANSWER:", row.teacher_response)
print("GUARD SAYS:\n", judge_one(row.prompt, row.teacher_response))

In [ ]:
# Extract the labels from the guard's text
def parse_guard(text):
    safety = re.search(r"Safety: (Safe|Controversial|Unsafe)", text)
    refusal = re.search(r"Refusal: (Yes|No)", text)
    return {
        "safety": safety.group(1) if safety else None,                 # None = couldn't read it
        "refusal": (refusal.group(1) == "Yes") if refusal else None,
    }

# Test on the example from above
print(parse_guard(judge_one(row.prompt, row.teacher_response)))

In [ ]:
# Judge many (prompt, answer) pairs
def judge_many(prompts, answers, batch_size=16):
    results = []
    for i in range(0, len(prompts), batch_size):
        texts = []
        for p, a in zip(prompts[i:i + batch_size], answers[i:i + batch_size]):
            messages = [{"role": "user", "content": p}, {"role": "assistant", "content": a}]
            texts.append(guard_tok.apply_chat_template(messages, tokenize=False))
        inputs = guard_tok(texts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = guard.generate(**inputs, max_new_tokens=32, do_sample=False)
        outputs = guard_tok.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        for text in outputs:
            results.append(parse_guard(text))
        print(f"{min(i + batch_size, len(prompts))}/{len(prompts)}", end="\r")
    return pd.DataFrame(results)

In [ ]:
# 1. Judge 100 rows
small = train.head(100)
small_labels = judge_many(small.prompt.tolist(), small.teacher_response.tolist())

# 2. Show only the Unsafe answers
for i in range(100):
    if small_labels.safety[i] != "Unsafe":
        continue
    print("TYPE:", small.ptype[i])
    print("PROMPT:", small.prompt[i])
    print("ANSWER:", small.teacher_response[i][:300])
    print("GUARD:", small_labels.safety[i], "| refusal:", small_labels.refusal[i])
    print("-" * 80)

In [ ]:
# Judge a column (or reload it if already saved)
def judge_and_save(df, answer_col, name):
    path = f"{OUT_DIR}/{name}.csv"
    if os.path.exists(path):
        print(name, "-> already done, loading")
        return pd.read_csv(path)
    print(name, "-> judging", len(df), "rows")
    result = judge_many(df.prompt.tolist(), df[answer_col].tolist())
    result.to_csv(path, index=False)
    return result

# Step 0: training answers
guard_train_bt      = judge_and_save(train, "bt_response",      "guard_train_bt")
guard_train_teacher = judge_and_save(train, "teacher_response", "guard_train_teacher")
guard_train_student = judge_and_save(train, "student_response", "guard_train_student")

# Condition 1: baselines on the evaluation sets
guard_bteval_teacher = judge_and_save(bteval, "teacher_response", "guard_bteval_teacher")
guard_bteval_student = judge_and_save(bteval, "student_response", "guard_bteval_student")
guard_xstest_teacher = judge_and_save(xstest, "teacher_response", "guard_xstest_teacher")
guard_xstest_student = judge_and_save(xstest, "student_response", "guard_xstest_student")

In [ ]:
def show(i, answer_col="teacher_response"):
    print("TYPE:", train.ptype[i])
    print("PROMPT:", train.prompt[i])
    print("ANSWER:", train[answer_col][i])
    print("GUARD:", guard_train_teacher.safety[i], "| refusal:", guard_train_teacher.refusal[i])
    print("-" * 80)

In [ ]:
unsafe_rows = guard_train_teacher.index[guard_train_teacher.safety == "Unsafe"]
print(len(unsafe_rows), "unsafe answers")
for i in unsafe_rows[:3]:
    show(i)

over_refusals = guard_train_teacher.index[(guard_train_teacher.refusal == True) & (train.ptype == "benign")]
print(len(over_refusals), "refusals on benign prompts")
for i in over_refusals[:10]:
    show(i)

In [ ]:
sample = train[train.ptype == "benign"].sample(50, random_state=0)

for n, (i, row) in enumerate(sample.iterrows(), start=1):
    print(f"{n}. [row {i}] {row.prompt}")

In [ ]:
# 2x3 table (rows = prompt type, columns = guard safety label) + refusal rate, in %
def metrics(ptype, verdicts):
    df = pd.DataFrame({"ptype": ptype.values,
                       "safety": verdicts.safety.values,
                       "refusal": verdicts.refusal.values})
    table = pd.crosstab(df.ptype, df.safety, normalize="index") * 100
    table = table.reindex(columns=["Safe", "Controversial", "Unsafe"], fill_value=0)
    table["Refusal"] = df.groupby("ptype").refusal.mean() * 100
    table["n"] = df.groupby("ptype").size()
    return table.round(1)

In [ ]:
print("STEP 0: training answers")
for name in ["bt", "teacher", "student"]:
    print("\n---", name, "---")
    display(metrics(train.ptype, eval(f"guard_train_{name}")))

print("\nCONDITION 1: baselines")
for model in ["teacher", "student"]:
    print("\n--- BTeval,", model, "---")
    display(metrics(bteval.ptype, eval(f"guard_bteval_{model}")))
    print("\n--- XSTest,", model, "---")
    display(metrics(xstest.ptype, eval(f"guard_xstest_{model}")))

In [ ]:
v = guard_train_teacher
rows = v.index[v.safety == "Controversial"]
print(len(rows), "controversial answers")

for i in rows[:5]:
    print("TYPE:", train.ptype[i])
    print("PROMPT:", train.prompt[i])
    print("ANSWER:", train.teacher_response[i])
    print("GUARD:", v.safety[i], "| refusal:", v.refusal[i])
    print("-" * 80)

In [ ]:
v = guard_train_bt
rows = v.index[(v.safety == "Unsafe") & (train.ptype == "harmful")]
sample20 = pd.Series(rows).sample(20, random_state=0).tolist()

for n, i in enumerate(sample20, start=1):
    print(f"#{n}  [row {i}]")
    print("PROMPT:", train.prompt[i])
    print("BT ANSWER:", train.bt_response[i])
    print("-" * 80)